# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ektam9931/flyrank-ml-week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### Contract

- **Unit of analysis:** One row represents observed daily performance for a content item (`content_hash_id`) for a client (`client_hash_id`) on a specific report date (`report_date`). The dataset contains a small number of duplicate date-client-content combinations, so this grain is treated as the observed record structure rather than assuming every combination is unique.

- **Table:** `fact_content_daily_performance`, using the March 2026 slice for development.

- **Time window:** March 1, 2026 through March 31, 2026 (`month = '2026-03'`).

- **Prediction target:** Expected content performance, using observed engagement/traffic outcomes as the performance proxy. The exact target will be defined from information available after the prediction point.

- **Excluded:** The June 2026 `_sample` table is deliberately excluded from development because it represents the final month and should be treated as a sealed outcome/test window.

In [45]:
%pip -q install duckdb

In [46]:
from google.colab import userdata
from huggingface_hub import login
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

print("Hugging Face connected ✅")
print("DuckDB version:", duckdb.__version__)

Hugging Face connected ✅
DuckDB version: 1.3.2


In [47]:
con = duckdb.connect()

con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN ?
)
""", [HF_TOKEN])

print("DuckDB connected to Hugging Face ✅")

DuckDB connected to Hugging Face ✅


In [48]:

import requests

url = "https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main"
headers = {"Authorization": f"Bearer {HF_TOKEN}"}

r = requests.get(url, headers=headers)
r.raise_for_status()

files = r.json()
for f in files:
    print(f["path"])

fact_content_daily_performance
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance_sample.parquet
fact_content_query_90d.parquet


In [49]:
query = """
DESCRIBE
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
"""

columns = con.execute(query).df()
columns

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [50]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS unique_grain_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
"""

con.execute(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_rows
0,78835655,78829265


In [51]:
query = """
SELECT
    COUNT(*) AS march_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

con.execute(query).df()

,march_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [52]:
query = """
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

con.execute(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


## 2. Fields: feature / label / context / excluded

### Fields

**Features**
- `gsc_impressions` — historical search impressions.
- `gsc_clicks` — historical search clicks.
- `gsc_avg_position` — historical average search position.
- `ga4_sessions` — historical sessions.
- `ga4_engaged_sessions` — historical engaged sessions.

**Label**
- `future_gsc_clicks` — future organic Google Search clicks used as the performance proxy; it represents an outcome that would only be known after the prediction point.

**Context**
- `report_date` — identifies the observation date.
- `month` — identifies the development month.
- `client_hash_id` — identifies the client.
- `content_hash_id` — identifies the content item.
- `gsc_data_available` — indicates whether GSC data is available.
- `ga4_data_available` — indicates whether GA4 data is available.

**Excluded**
- The June 2026 `_sample` table is excluded from development because it represents the final month and should remain a sealed outcome/test window.

In [53]:
query = """
SELECT
    report_date,
    SUM(gsc_clicks) AS total_gsc_clicks,
    SUM(gsc_impressions) AS total_gsc_impressions,
    SUM(ga4_sessions) AS total_ga4_sessions,
    SUM(ga4_engaged_sessions) AS total_ga4_engaged_sessions
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY report_date
ORDER BY report_date
LIMIT 10
"""

con.execute(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,total_gsc_clicks,total_gsc_impressions,total_ga4_sessions,total_ga4_engaged_sessions
0,2026-03-01,24165.0,7671976.0,8044.0,538.0
1,2026-03-02,26946.0,8383856.0,9375.0,726.0
2,2026-03-03,26849.0,9016431.0,13511.0,914.0
3,2026-03-04,26943.0,9538450.0,13437.0,896.0
4,2026-03-05,26224.0,9319687.0,15740.0,852.0
5,2026-03-06,23671.0,8548065.0,15432.0,815.0
6,2026-03-07,21973.0,7610356.0,13858.0,723.0
7,2026-03-08,22452.0,7809866.0,13400.0,686.0
8,2026-03-09,28098.0,9699465.0,17240.0,905.0
9,2026-03-10,28096.0,9575169.0,22816.0,1024.0


## 3. Verify it with queries (grain, counts, missing values, windows)

### Five Features — Available When?

| Feature | Available when? |
|---|---|
| `gsc_impressions` | Known at the decision moment because these Google Search impressions were already recorded during the March observation window. |
| `gsc_clicks` | Known at the decision moment because these organic Google Search clicks were already observed during March. |
| `gsc_avg_position` | Known at the decision moment because the average Google Search position is calculated from observations recorded during March. |
| `ga4_sessions` | Known at the decision moment because these sessions were already recorded in Google Analytics during March. |
| `ga4_engaged_sessions` | Known at the decision moment because these engaged sessions were already recorded in Google Analytics during March. |

**Prediction setup:** March 2026 features → April 2026 `future_gsc_clicks`.

The April outcome is deliberately excluded from the feature frame because it would not be known when making the prediction.

In [62]:

label_query = """
WITH april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS future_gsc_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.*,
    a.future_gsc_clicks
FROM features f
LEFT JOIN april a
    USING (client_hash_id, content_hash_id)
"""

model_df = con.execute(label_query).df()

model_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,future_gsc_clicks
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,0.0,8.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,0.0,2.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,3.0,0.0,4.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,0.0,8.0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,7.0,0.0,0.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,future_gsc_clicks
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,0.0,8.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,0.0,2.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,3.0,0.0,4.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,0.0,8.0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,7.0,0.0,0.0


In [63]:
print("Rows:", len(model_df))
print("Columns:", model_df.columns.tolist())
print("Missing future labels:", model_df["future_gsc_clicks"].isna().sum())

Rows: 331437
Columns: ['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'future_gsc_clicks']
Missing future labels: 1
Rows: 331437
Columns: ['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'future_gsc_clicks']
Missing future labels: 1


In [56]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

model_df_clean = model_df.dropna(subset=["future_gsc_clicks"]).copy()

print("Original rows:", len(model_df))
print("Rows with valid future labels:", len(model_df_clean))
print("Rows removed:", len(model_df) - len(model_df_clean))

X = model_df_clean[feature_cols].fillna(0)
y = model_df_clean["future_gsc_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

baseline_model = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_r2 = r2_score(y_test, baseline_pred)

print("Honest baseline MAE:", baseline_mae)
print("Honest baseline R²:", baseline_r2)

Original rows: 331437
Rows with valid future labels: 331436
Rows removed: 1
Honest baseline MAE: 1.3318894440350182
Honest baseline R²: 0.8308956439573378


In [57]:


leaky_df = model_df_clean.copy()

leaky_df["leaked_future_clicks"] = leaky_df["future_gsc_clicks"]

leaky_features = feature_cols + ["leaked_future_clicks"]

X_leaky = leaky_df[leaky_features].fillna(0)
y_leaky = leaky_df["future_gsc_clicks"]

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky,
    y_leaky,
    test_size=0.2,
    random_state=42
)

leaky_model = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

leaky_model.fit(X_train_leaky, y_train_leaky)

leaky_pred = leaky_model.predict(X_test_leaky)

leaky_mae = mean_absolute_error(y_test_leaky, leaky_pred)
leaky_r2 = r2_score(y_test_leaky, leaky_pred)

print("Leaky baseline MAE:", leaky_mae)
print("Leaky baseline R²:", leaky_r2)

Leaky baseline MAE: 0.0034015206372194078
Leaky baseline R²: 0.999577270300465


### Deliberate Leakage Experiment

I intentionally added `leaked_future_clicks`, which is derived directly from the April target `future_gsc_clicks`, to the feature set.

The resulting score became suspiciously strong because the model was given information that would only exist after the March prediction moment. This is target leakage: the feature contains the outcome we are trying to predict.

The experiment demonstrates why a high model score is not automatically evidence of a useful model. A feature can make prediction appear almost perfect simply because it contains future information.

**Leaky result:**

- MAE: Leaky baseline MAE: 0.002538015930485161

- R²: Leaky baseline R²: 0.9997929792122859

The leaked feature is not valid for prediction and must be removed from the final feature frame.

In [58]:


final_features = model_df[feature_cols].copy()

print("Final feature columns:")
print(final_features.columns.tolist())

print(
    "\nLeaked feature present:",
    "leaked_future_clicks" in final_features.columns
)

Final feature columns:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']

Leaked feature present: False


### Leakage Conclusion

The honest feature set uses only information available during the March 2026 observation window. The deliberate leakage experiment showed that including `future_gsc_clicks` as a feature produces an artificially strong result because the model is directly given future outcome information.

I therefore removed `leaked_future_clicks` from the final feature frame.

**Final features retained:**

1. `gsc_impressions`
2. `gsc_clicks`
3. `gsc_avg_position`
4. `ga4_sessions`
5. `ga4_engaged_sessions`

**Prediction target:** `future_gsc_clicks` — April 2026 organic Google Search clicks.

The honest baseline score is the valid reference. The leaky score is used only to demonstrate the leakage trap.

In [59]:
query = """
SELECT
    month,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT report_date) AS days
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month IN ('2026-03', '2026-04')
GROUP BY month
ORDER BY month
"""

con.execute(query).df()

,month,first_date,last_date,days
0,2026-03,2026-03-01,2026-03-31,31
1,2026-04,2026-04-01,2026-04-30,30


In [60]:
query = """
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS march_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    COUNT(*) AS march_content_rows,
    COUNT(april.april_clicks) AS rows_with_future_label,
    MIN(april.april_clicks) AS min_future_clicks,
    MAX(april.april_clicks) AS max_future_clicks
FROM march
LEFT JOIN april
    USING (client_hash_id, content_hash_id)
"""

con.execute(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_content_rows,rows_with_future_label,min_future_clicks,max_future_clicks
0,331437,331436,0.0,7434.0


In [61]:
feature_query = """
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_sessions) AS ga4_sessions,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY client_hash_id, content_hash_id
"""

features = con.execute(feature_query).df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,0.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,0.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,3.0,0.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,0.0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,7.0,0.0


## 4. Data limits

### Limitation — Temporal Generalization

This feature slice uses March 2026 as the observation window and April 2026 as the future outcome window. Content performance can change across months because of seasonality, search demand, algorithm changes, and other external factors. Therefore, results from this single March → April transition may not generalize to every future month.

For this assignment, March is used as the development month and the later June 2026 month remains sealed rather than being used to develop the label or features.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### ML-04 Self-check

- [x] One row represents one client-content pair aggregated over the March 2026 observation window.
- [x] The warehouse source is `fact_content_daily_performance`.
- [x] March 2026 is used as the development/feature month.
- [x] April 2026 organic clicks are used as the future performance proxy.
- [x] Exactly five features are used.
- [x] Each feature has an “Available when?” explanation.
- [x] Three verification queries were run with visible outputs.
- [x] Availability verification uses `IS TRUE`.
- [x] A deliberate label-derived leakage feature was added and evaluated.
- [x] The leakage was identified and explained.
- [x] The leaked feature was removed from the final feature frame.
- [x] One limitation of the slice is documented.
- [x] June 2026 was not used to develop the label logic.